# Noise Removal, Filter Comparison and Object Analysis
Corrupt images with 4 noise types, denoise with Mean / Gaussian / Median / Bilateral filters,
compare with MSE / RMSE / PSNR / SSIM / edge preservation / runtime, then segment, count and
describe objects and see how filtering changes counting accuracy and shape descriptors.

Run all cells top to bottom. Inputs are obtained automatically; results go to `output_imgs/`.

In [ ]:
!pip install -q numpy opencv-python scikit-image pandas matplotlib requests

In [ ]:
import os, time, warnings, zipfile
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import requests
from skimage.metrics import structural_similarity
from skimage.feature import peak_local_max
from skimage.segmentation import watershed, mark_boundaries
from skimage.measure import regionprops

warnings.filterwarnings("ignore")
SEED = 42
OUT = "output_imgs"
SUBDIRS = ["original", "noisy", "filtered", "segmentation", "comparisons", "plots"]
NOISES = ["gaussian", "salt_pepper", "speckle", "poisson"]
FILTERS = ["Mean", "Gaussian", "Median", "Bilateral"]
NOISE_LABEL = {"gaussian": "Gaussian", "salt_pepper": "Salt & Pepper",
               "speckle": "Speckle", "poisson": "Poisson"}


def make_dirs():
    for d in SUBDIRS:
        os.makedirs(os.path.join(OUT, d), exist_ok=True)


def to_gray(img):
    return img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def save_img(folder, name, img):
    cv2.imwrite(os.path.join(OUT, folder, name + ".png"), img)

## 1. Input images (download, else bundled, else synthetic)

In [ ]:
def make_synthetic(seed, n_objects, color=True, size=320):
    """Dark background with bright circles/ellipses, plus one touching pair (tests watershed)."""
    rng = np.random.default_rng(seed)
    img = np.full((size, size, 3), 45, np.uint8)
    placed = []  # (x, y, r)

    def free(x, y, r, ignore=None):
        return all(np.hypot(x - px, y - py) > r + pr + 8
                   for i, (px, py, pr) in enumerate(placed) if i != ignore)

    def draw(x, y, r, ellipse):
        col = tuple(int(c) for c in rng.integers(150, 256, 3))
        if ellipse:
            axes = (int(r), max(8, int(r * rng.uniform(0.55, 0.8))))
            cv2.ellipse(img, (x, y), axes, float(rng.integers(0, 180)), 0, 360, col, -1)
        else:
            cv2.circle(img, (x, y), int(r), col, -1)

    tries = 0
    while len(placed) < n_objects - 2 and tries < 5000:
        tries += 1
        r = int(rng.integers(12, 26))
        x, y = int(rng.integers(r + 5, size - r - 5)), int(rng.integers(r + 5, size - r - 5))
        if free(x, y, r):
            placed.append((x, y, r))
            draw(x, y, r, rng.random() < 0.4)
    # touching pair
    for _ in range(2000):
        r = int(rng.integers(14, 22))
        x, y = int(rng.integers(r + 5, size - 2 * r - 10)), int(rng.integers(r + 5, size - 2 * r - 10))
        x2, y2 = x + int(1.7 * r), y
        if free(x, y, r) and free(x2, y2, r):
            placed += [(x, y, r), (x2, y2, r)]
            draw(x, y, r, False)
            draw(x2, y2, r, False)
            break
    return img if color else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def limit_size(img, max_side=400):
    h, w = img.shape[:2]
    s = max_side / max(h, w)
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else img


def download_or_generate_images():
    images = {}
    # 1) try downloading a public sample (coins) with requests
    try:
        url = "https://raw.githubusercontent.com/scikit-image/scikit-image/main/skimage/data/coins.png"
        r = requests.get(url, timeout=8)
        r.raise_for_status()
        img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_UNCHANGED)
        if img is None:
            raise ValueError("could not decode download")
        images["coins_downloaded"] = limit_size(img[..., :3] if img.ndim == 3 and img.shape[2] > 3 else img)
        print("Downloaded sample image: coins")
    except Exception as e:
        print("Download failed (" + type(e).__name__ + "); trying bundled skimage image")
        try:
            from skimage import data
            images["coins_bundled"] = limit_size(np.asarray(data.coins(), np.uint8))
        except Exception as e2:
            print("Bundled image unavailable (" + type(e2).__name__ + "); using synthetic only")
    # 2) synthetic images are always generated so the experiment always runs
    images["synthetic_color"] = make_synthetic(SEED, 11, color=True)
    images["synthetic_gray"] = make_synthetic(SEED + 1, 14, color=False)
    for name, im in images.items():
        save_img("original", name, im)
    print("Images:", list(images))
    return images

## 2. Noise and filters

In [ ]:
def add_noise(img, kind, rng):
    f = img.astype(np.float64)
    if kind == "gaussian":
        out = f + rng.normal(0, 25, f.shape)
    elif kind == "salt_pepper":
        out = f.copy()
        m = rng.random(img.shape[:2])
        out[m < 0.025] = 0
        out[m > 0.975] = 255
    elif kind == "speckle":
        out = f + f * rng.normal(0, np.sqrt(0.1), f.shape)
    elif kind == "poisson":
        peak = 30.0
        out = rng.poisson(f / 255.0 * peak) / peak * 255.0
    else:
        raise ValueError("unknown noise: " + kind)
    return np.clip(out, 0, 255).astype(np.uint8)


def apply_filters(noisy):
    """Return {name: (filtered_image, runtime_ms)}; runtime is the best of 3 runs."""
    fns = {
        "Mean": lambda x: cv2.blur(x, (5, 5)),
        "Gaussian": lambda x: cv2.GaussianBlur(x, (5, 5), 1.5),
        "Median": lambda x: cv2.medianBlur(x, 5),
        "Bilateral": lambda x: cv2.bilateralFilter(x, 9, 75, 75),
    }
    out = {}
    for name, fn in fns.items():
        best = float("inf")
        for _ in range(3):
            t0 = time.perf_counter()
            res = fn(noisy)
            best = min(best, (time.perf_counter() - t0) * 1000)
        out[name] = (res, best)
    return out

## 3. Quality and edge metrics

In [ ]:
def calculate_edge_preservation(clean, test):
    """F1 overlap of Canny edge maps (3x3 tolerance). 1.0 = identical edges."""
    ec = cv2.Canny(to_gray(clean), 80, 160) > 0
    et = cv2.Canny(to_gray(test), 80, 160) > 0
    k = np.ones((3, 3), np.uint8)
    ec_d = cv2.dilate(ec.astype(np.uint8), k) > 0
    et_d = cv2.dilate(et.astype(np.uint8), k) > 0
    precision = (et & ec_d).sum() / max(et.sum(), 1)
    recall = (ec & et_d).sum() / max(ec.sum(), 1)
    return 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0


def calculate_metrics(clean, test):
    mse = float(np.mean((clean.astype(np.float64) - test.astype(np.float64)) ** 2))
    rmse = float(np.sqrt(mse))
    psnr = 100.0 if mse < 1e-10 else min(100.0, 10 * np.log10(255.0 ** 2 / mse))
    ssim = structural_similarity(clean, test, data_range=255,
                                 channel_axis=-1 if clean.ndim == 3 else None)
    return mse, rmse, float(psnr), float(ssim), float(calculate_edge_preservation(clean, test))

## 4. Segmentation, counting, descriptors

In [ ]:
def detect_invert(img):
    """Decide polarity once on the CLEAN image so noisy/filtered use the same rule."""
    blur = cv2.GaussianBlur(to_gray(img), (5, 5), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    border = np.concatenate([mask[0], mask[-1], mask[:, 0], mask[:, -1]])
    return bool(border.mean() > 127)


def segment_objects(img, invert=False):
    """Gaussian blur -> Otsu -> morphology -> distance-transform watershed -> label image."""
    gray = to_gray(img)
    h, w = gray.shape
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if invert:
        mask = 255 - mask
    k = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    if mask.max() == 0:
        return np.zeros((h, w), np.int32)
    dist = cv2.GaussianBlur(cv2.distanceTransform(mask, cv2.DIST_L2, 5), (0, 0), 2)
    _, cc = cv2.connectedComponents(mask)
    peaks = peak_local_max(dist, min_distance=8, threshold_abs=2, labels=cc, exclude_border=False)
    if len(peaks) == 0:
        return np.zeros((h, w), np.int32)
    markers = np.zeros((h, w), np.int32)
    for i, (r, c) in enumerate(peaks, 1):
        markers[r, c] = i
    labels = watershed(-dist, markers, mask=mask > 0)
    areas = np.bincount(labels.ravel())
    keep = areas >= max(30, int(0.001 * h * w))
    keep[0] = False
    labels = np.where(keep[labels], labels, 0)
    _, labels = np.unique(labels, return_inverse=True)
    return labels.reshape(h, w).astype(np.int32)


def extract_descriptors(labels):
    objs = []
    for p in regionprops(labels):
        r0, c0, r1, c1 = p.bbox
        perim = p.perimeter
        objs.append({
            "area": float(p.area), "perimeter": float(perim),
            "centroid": (float(p.centroid[0]), float(p.centroid[1])),
            "bbox": (int(r0), int(c0), int(r1), int(c1)),
            "aspect_ratio": (c1 - c0) / max(r1 - r0, 1),
            "circularity": float(min(4 * np.pi * p.area / perim ** 2, 1.5)) if perim > 0 else 0.0,
            "eccentricity": float(p.eccentricity), "solidity": float(p.solidity)})
    return objs


def compare_descriptors(ref, det):
    """Mean relative error of area/circularity/solidity/aspect_ratio vs the clean reference.
    Unmatched reference objects count as error 1.0. Returns (error, matched_fraction)."""
    if not ref:
        return 0.0, 1.0
    keys = ["area", "circularity", "solidity", "aspect_ratio"]
    errs, matched = [], 0
    for r in ref:
        best, bd = None, 1e9
        for d in det:
            dist = np.hypot(r["centroid"][0] - d["centroid"][0], r["centroid"][1] - d["centroid"][1])
            if dist < bd:
                best, bd = d, dist
        if best is not None and bd <= max(10.0, np.sqrt(r["area"] / np.pi)):
            matched += 1
            errs.append(np.mean([min(abs(best[k] - r[k]) / max(abs(r[k]), 1e-9), 1.0) for k in keys]))
        else:
            errs.append(1.0)
    return float(np.mean(errs)), matched / len(ref)

## 5. Run the experiment

In [ ]:
def grid_figure(items, path, suptitle):
    """items: list of 6 (title, image, cmap)."""
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, (title, im, cmap) in zip(axes.ravel(), items):
        ax.imshow(im, cmap=cmap)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    fig.suptitle(suptitle)
    fig.tight_layout()
    fig.savefig(path, dpi=90)
    plt.close(fig)


def run_experiment(images):
    rng = np.random.default_rng(SEED)
    rows = []
    for name, clean in images.items():
        try:
            print("Processing", name)
            inv = detect_invert(clean)
            ref_labels = segment_objects(clean, inv)
            ref_desc = extract_descriptors(ref_labels)
            true_count = len(ref_desc)
            print("  clean-image object count:", true_count)
            overlay = lambda im, lb: mark_boundaries(to_rgb(im), lb, color=(1, 0, 0))
            grid_figure([("Original", to_rgb(clean), None),
                         ("Segmentation (count=%d)" % true_count, overlay(clean, ref_labels), None),
                         ("Edges (Canny)", cv2.Canny(to_gray(clean), 80, 160), "gray"),
                         ("Binary mask", (ref_labels > 0).astype(float), "gray"),
                         ("", np.zeros((2, 2)), "gray"), ("", np.zeros((2, 2)), "gray")],
                        os.path.join(OUT, "segmentation", name + "_original.png"), name + " (clean reference)")
            for noise in NOISES:
                noisy = add_noise(clean, noise, rng)
                save_img("noisy", name + "_" + noise, noisy)
                variants = {"NoFilter": (noisy, 0.0)}
                variants.update(apply_filters(noisy))
                shown, edges, segs = [("Original", to_rgb(clean), None)], [("Original edges", cv2.Canny(to_gray(clean), 80, 160), "gray")], []
                for fname, (img, ms) in variants.items():
                    if fname != "NoFilter":
                        save_img("filtered", "%s_%s_%s" % (name, noise, fname), img)
                    mse, rmse, psnr, ssim, edge = calculate_metrics(clean, img)
                    labels = segment_objects(img, inv)
                    desc = extract_descriptors(labels)
                    derr, dfrac = compare_descriptors(ref_desc, desc)
                    err = abs(len(desc) - true_count)
                    acc = max(0.0, 1 - err / true_count) if true_count > 0 else (1.0 if err == 0 else 0.0)
                    rows.append({"image": name, "noise_type": noise, "filter": fname, "mse": mse, "rmse": rmse,
                                 "psnr": psnr, "ssim": ssim, "edge_preservation": edge, "runtime_ms": ms,
                                 "true_count": true_count, "detected": len(desc), "abs_error": err,
                                 "count_accuracy": acc, "desc_error": derr, "desc_matched_frac": dfrac})
                    label = "Noisy" if fname == "NoFilter" else fname
                    shown.append(("%s\nPSNR %.1f SSIM %.2f" % (label, psnr, ssim), to_rgb(img), None))
                    edges.append((label + " edges", cv2.Canny(to_gray(img), 80, 160), "gray"))
                    segs.append(("%s: %d objects (true %d)" % (label, len(desc), true_count), overlay(img, labels), None))
                # comparison grid: Original, Noisy, Mean, Gaussian, Median, Bilateral
                title = "%s - %s noise" % (name, NOISE_LABEL[noise])
                grid_figure(shown, os.path.join(OUT, "comparisons", "%s_%s.png" % (name, noise)), title)
                grid_figure(edges, os.path.join(OUT, "comparisons", "%s_%s_edges.png" % (name, noise)), title + " (edge maps)")
                grid_figure(segs + [("", np.zeros((2, 2)), "gray")],
                            os.path.join(OUT, "segmentation", "%s_%s.png" % (name, noise)), title + " (segmentation)")
        except Exception as e:
            print("  ERROR processing %s: %s: %s" % (name, type(e).__name__, e))
    df = pd.DataFrame(rows)
    df.round(4).to_csv(os.path.join(OUT, "results.csv"), index=False)
    print("Saved", os.path.join(OUT, "results.csv"), "with", len(df), "rows")
    return df

## 6. Plots

In [ ]:
def create_plots(df):
    d = df[df["filter"] != "NoFilter"]
    metrics = [("psnr", "PSNR (dB, higher better)"), ("ssim", "SSIM (higher better)"),
               ("edge_preservation", "Edge preservation F1 (higher better)"),
               ("runtime_ms", "Runtime (ms, lower better)"),
               ("count_accuracy", "Object-count accuracy (higher better)"),
               ("desc_error", "Descriptor error (lower better)")]
    for col, title in metrics:
        pv = d.pivot_table(index="noise_type", columns="filter", values=col, aggfunc="mean")
        pv = pv.reindex([n for n in NOISES if n in pv.index])[[f for f in FILTERS if f in pv.columns]]
        pv.index = [NOISE_LABEL[i] for i in pv.index]
        ax = pv.plot(kind="bar", figsize=(8, 5), rot=0)
        ax.set_title(title)
        ax.set_xlabel("")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT, "plots", col + ".png"), dpi=100)
        plt.close()
    print("Saved plots to", os.path.join(OUT, "plots"))

## 7. Best-filter analysis and final report

In [ ]:
def best(m, col, higher=True, tie=None, tie_higher=False):
    m = m.round(6)
    cols, asc = [col], [not higher]
    if tie:
        cols.append(tie)
        asc.append(not tie_higher)
    return m.sort_values(cols, ascending=asc).index[0]


def generate_final_report(df):
    d = df[df["filter"] != "NoFilter"]
    L = []
    P = L.append
    norm = lambda s: (s - s.min()) / (s.max() - s.min()) if s.max() - s.min() > 1e-12 else s * 0 + 1
    per_noise, winners = {}, {}
    for noise in [n for n in NOISES if n in set(d["noise_type"])]:
        m = d[d["noise_type"] == noise].groupby("filter")[
            ["psnr", "ssim", "edge_preservation", "runtime_ms", "count_accuracy", "desc_error"]].mean()
        score = (0.3 * norm(m.psnr) + 0.3 * norm(m.ssim) + 0.3 * norm(m.edge_preservation)
                 + 0.1 * (1 - norm(m.runtime_ms)))
        per_noise[noise] = m
        winners[noise] = {
            "overall": score.idxmax(), "psnr": best(m, "psnr"), "ssim": best(m, "ssim"),
            "edge": best(m, "edge_preservation"), "count": best(m, "count_accuracy", True, "desc_error", False),
            "desc": best(m, "desc_error", False, "count_accuracy", True)}

    P("BEST OVERALL FILTER PER NOISE TYPE (0.3 PSNR + 0.3 SSIM + 0.3 edge + 0.1 speed)")
    P("%-16s %s" % ("Noise Type", "Best Overall Filter"))
    P("-" * 36)
    for n, w in winners.items():
        P("%-16s %s" % (NOISE_LABEL[n], w["overall"]))

    P("")
    P("OBJECT COUNTING SUMMARY (counts summed over all images, accuracy = mean)")
    P("%-15s %-10s %10s %10s %10s" % ("Noise", "Filter", "True Count", "Detected", "Accuracy"))
    P("-" * 60)
    for n in winners:
        for f in ["NoFilter"] + FILTERS:
            g = df[(df["noise_type"] == n) & (df["filter"] == f)]
            if len(g):
                P("%-15s %-10s %10d %10d %9.0f%%" % (NOISE_LABEL[n], f, g.true_count.sum(), g.detected.sum(),
                                                   100 * g.count_accuracy.mean()))

    overall = d.groupby("filter")[["psnr", "ssim", "edge_preservation", "runtime_ms",
                                   "count_accuracy", "desc_error"]].mean()
    P("")
    P("FINAL ANALYSIS")
    P("1. Best image quality per noise type (PSNR / SSIM winners):")
    for n, w in winners.items():
        P("   - %-14s PSNR: %-10s SSIM: %s" % (NOISE_LABEL[n], w["psnr"], w["ssim"]))
    P("2. Best edge preservation: overall %s (%.3f); per noise: %s" % (
        best(overall, "edge_preservation"), overall.edge_preservation.max(),
        ", ".join("%s=%s" % (NOISE_LABEL[n], w["edge"]) for n, w in winners.items())))
    P("3. Fastest filter: %s (mean %.3f ms; slowest %s %.3f ms)" % (
        best(overall, "runtime_ms", False), overall.runtime_ms.min(),
        overall.runtime_ms.idxmax(), overall.runtime_ms.max()))
    P("4. Most accurate object count: overall %s (%.1f%%); per noise: %s" % (
        best(overall, "count_accuracy", True, "desc_error", False), 100 * overall.count_accuracy.max(),
        ", ".join("%s=%s" % (NOISE_LABEL[n], w["count"]) for n, w in winners.items())))
    P("5. Best shape-descriptor preservation: overall %s (error %.3f); per noise: %s" % (
        best(overall, "desc_error", False, "count_accuracy", True), overall.desc_error.min(),
        ", ".join("%s=%s" % (NOISE_LABEL[n], w["desc"]) for n, w in winners.items())))
    same = [n for n, w in winners.items() if w["psnr"] == w["count"] == w["desc"]]
    diff = [n for n in winners if n not in same]
    P("6. Is the best-PSNR filter always best for object analysis?")
    P("   PSNR winner equals both count and descriptor winner for %d of %d noise types." % (len(same), len(winners)))
    if diff:
        P("   Differs for: " + "; ".join("%s (PSNR=%s, count=%s, descriptors=%s)" % (
            NOISE_LABEL[n], winners[n]["psnr"], winners[n]["count"], winners[n]["desc"]) for n in diff))
        P("   => No: best PSNR does not necessarily give the best object counts or shapes.")
    else:
        P("   => In this run the best-PSNR filter also won object analysis for every noise type.")
    P("")
    P("(Counts use the clean-image detection as reference. Ties are broken by descriptor error.)")
    text = "\n".join(L)
    print(text)
    with open(os.path.join(OUT, "final_summary.txt"), "w", encoding="utf-8") as f:
        f.write(text)
    return text

## 8. Run everything

In [ ]:
np.random.seed(SEED)
make_dirs()
images = download_or_generate_images()
df = run_experiment(images)
if len(df):
    create_plots(df)
    generate_final_report(df)
else:
    print("No results were produced; check the error messages above.")

## 9. View a few results and download everything

In [ ]:
from IPython.display import Image as IPImage, display
for sub, pattern in [("comparisons", "_gaussian.png"), ("segmentation", "_salt_pepper.png"), ("plots", "psnr.png")]:
    folder = os.path.join(OUT, sub)
    hits = sorted(f for f in os.listdir(folder) if f.endswith(pattern))
    if hits:
        display(IPImage(os.path.join(folder, hits[-1])))

with zipfile.ZipFile("output_imgs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files:
            p = os.path.join(root, fn)
            z.write(p, p)
try:
    from google.colab import files
    files.download("output_imgs.zip")
except ImportError:
    print("Zipped results to output_imgs.zip")